In [2]:
from pinecone import Pinecone
import os
import getpass

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API key: ")

pinecone_api_key = os.environ.get("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

In [4]:
index_name = "ncl-full-itineraries"  

existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if index_name in existing_indexes:
    index = pc.Index(index_name)
else:
    raise ValueError(f"Index '{index_name}' does not exist. Please create it first.")

In [5]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [6]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [7]:
from uuid import uuid4
from langchain_core.documents import Document

# Budget ranges:
# economy: < $2000
# standard: $2000-$3500
# premium: $3500-$5000
# luxury: $5000-$8000
# ultra_luxury: > $8000

documents = [
    Document(
        page_content="I'm dreaming of a Caribbean getaway that combines relaxation with adventure. I want to spend my mornings lounging on pristine beaches with crystal clear waters, and my afternoons trying exciting water sports or exploring local islands. I'm looking for a balanced vacation that won't break the bank but still offers quality experiences. The warm tropical climate and ocean breezes sound perfect for escaping daily stress.",
        metadata={
            "id": 1,
            "name": "Sunny Caribbean Cruise",
            "weather": "tropical",
            "destination": "beach",
            "activities": ["relaxation", "adventure"],
            "budget": "standard",
            "cost_per_guest": 2499,
            "image": "https://img.freepik.com/free-photo/beautiful-asian-female-woman-relax-casual-leisure-peaceful-moment-cruise-deck-vacation-summer-time_609648-740.jpg",
            "description": "Enjoy the warm sun ☀️ and beautiful beaches 🏖️ of the Caribbean.",
            "sailing dates": ["06/15/2025", "06/22/2025"]
        }
    ),
    Document(
        page_content="I want to explore the majestic glaciers and wilderness of Alaska. I'm passionate about wildlife photography and dream of capturing images of whales breaching, bears fishing for salmon, and eagles soaring overhead. I'm seeking a premium expedition that combines adventure with comfortable accommodations. Expert naturalist guides and educational programs about the local ecosystem would make this trip perfect.",
        metadata={
            "id": 2,
            "name": "Alaskan Glacier Expedition",
            "weather": "polar",
            "destination": "mountain",
            "activities": ["wildlife", "adventure"],
            "budget": "premium",
            "cost_per_guest": 4599,
            "image": "https://img.freepik.com/free-photo/couple-traveling-together-country-side_23-2149406524.jpg",
            "description": "Explore the rugged beauty of Alaska's glaciers ❄️🏔️.",
            "sailing dates": ["07/01/2025", "07/10/2025"]
        }
    ),
    Document(
        page_content="I'm longing to immerse myself in the rich cultural tapestry of the Mediterranean. I want to explore ancient ruins, taste authentic local cuisine, and learn about the history of different civilizations. I'm looking for a balanced itinerary that combines guided cultural tours with enough free time to wander through charming coastal towns. Ideally, I'd like a mix of educational experiences and leisure time to enjoy the local lifestyle.",
        metadata={
            "id": 3,
            "name": "Mediterranean Cultural Tour",
            "weather": "temperate",
            "destination": "city",
            "activities": ["cultural", "relaxation"],
            "budget": "standard",
            "cost_per_guest": 2999,
            "image": "https://img.freepik.com/free-photo/mother-her-daughter-eating-harvested-olives-field_23-2147907340.jpg",
            "description": "Discover the rich history 🏛️ and culture 🎭 of the Mediterranean.",
            "sailing dates": ["09/05/2025", "09/15/2025"]
        }
    ),
    Document(
        page_content="I've always dreamed of experiencing an authentic African safari adventure. I want to witness the great migration, see the big five in their natural habitat, and learn about conservation efforts. I'm looking for a premium experience with expert guides who can share their knowledge about the wildlife and ecosystem. Comfortable lodging and professional photography opportunities are important to me. I'd love to combine game drives with cultural visits to local communities.",
        metadata={
            "id": 4,
            "name": "African Safari Adventure",
            "weather": "tropical",
            "destination": "countryside",
            "activities": ["wildlife", "adventure"],
            "budget": "premium",
            "cost_per_guest": 4799,
            "image": "https://img.freepik.com/free-photo/beautiful-shot-group-african-wildebeests-grassy-plain_181624-27243.jpg",
            "description": "Experience the thrill of a safari 🦁🐘 in the African savannah.",
            "sailing dates": ["08/10/2025", "08/20/2025"]
        }
    ),
    Document(
        page_content="I want to experience the enchanting beauty of Japan during cherry blossom season. I'm traveling with my family and we're interested in experiencing traditional Japanese culture, from tea ceremonies to temple visits. We'd love to learn about local customs, try authentic cuisine, and maybe even participate in a traditional workshop. Looking for a trip that's educational for the children but still comfortable and engaging for adults.",
        metadata={
            "id": 5,
            "name": "Japanese Cherry Blossom Tour",
            "weather": "temperate",
            "destination": "city",
            "activities": ["cultural", "family"],
            "budget": "standard",
            "cost_per_guest": 3299,
            "image": "https://img.freepik.com/free-photo/row-cherry-blossom-tree-with-cherry-blossom-falling-petals-springtime-kyoto-japan_335224-1334.jpg",
            "description": "Witness the beauty of cherry blossoms 🌸 in Japan's historic cities.",
            "sailing dates": ["03/25/2025", "04/05/2025"]
        }
    ),
    Document(
        page_content="I'm looking for a luxurious winter sports vacation in the Swiss Alps. I want to experience world-class skiing during the day and indulge in spa treatments and gourmet dining in the evenings. I'd love to stay in a premium resort with stunning mountain views, access to top-tier skiing facilities, and wellness amenities. The perfect mix of active adventure and refined relaxation.",
        metadata={
            "id": 6,
            "name": "Skiing in the Swiss Alps",
            "weather": "temperate",
            "destination": "mountain",
            "activities": ["sports", "relaxation"],
            "budget": "luxury",
            "cost_per_guest": 6799,
            "image": "https://img.freepik.com/free-photo/beautiful-view-people-cycling-skiing-across-snowy-mountains-south-tyrol-dolomites-italy_181624-29926.jpg",
            "description": "Hit the slopes ⛷️ and enjoy the scenic beauty of the Swiss Alps 🏔️.",
            "sailing dates": ["01/15/2025", "01/25/2025"]
        }
    ),
    Document(
        page_content="My partner and I want to celebrate our anniversary with a romantic getaway to Paris. We're looking for an intimate experience that combines luxury accommodations with authentic Parisian culture. We dream of enjoying private wine tastings, candlelit dinners at elegant restaurants, and walks along the Seine. We'd love insider access to museums and cultural sites, plus time to discover hidden romantic spots in the city.",
        metadata={
            "id": 7,
            "name": "Romantic Getaway in Paris",
            "weather": "temperate",
            "destination": "city",
            "activities": ["romantic", "cultural"],
            "budget": "premium",
            "cost_per_guest": 4299,
            "image": "https://img.freepik.com/free-photo/couple-browsing-smartphones-date_23-2147744393.jpg",
            "description": "Spend a romantic weekend exploring the City of Love ❤️🗼.",
            "sailing dates": ["02/10/2025", "02/17/2025"]
        }
    ),
    Document(
        page_content="I'm seeking a transformative wellness retreat in Bali that focuses on holistic healing and spiritual renewal. I want to practice yoga in beautiful outdoor settings, learn meditation from experienced teachers, and try traditional Balinese healing treatments. I'm interested in organic cooking classes, mindfulness workshops, and connecting with like-minded travelers. Luxury accommodation in a serene setting is important to me.",
        metadata={
            "id": 8,
            "name": "Wellness Retreat in Bali",
            "weather": "tropical",
            "destination": "island",
            "activities": ["wellness", "relaxation"],
            "budget": "luxury",
            "cost_per_guest": 7299,
            "image": "https://img.freepik.com/free-photo/young-woman-with-body-positive-appearance-practicing-yoga-alone-deck-by-pool-tropical-island-bali-indonesia-sport-fitness-healthy-lifestyle-concept_1321-2876.jpg",
            "description": "Rejuvenate your body and soul 🧘‍♀️ at a luxurious Bali retreat.",
            "sailing dates": ["05/01/2025", "05/15/2025"]
        }
    ),
    Document(
        page_content="I want to embark on the ultimate polar expedition to Antarctica. I'm passionate about climate science and wildlife photography, and dream of capturing images of penguins, seals, and whales in their natural habitat. I'm seeking the most comprehensive and exclusive Antarctic experience available, with access to scientific research stations and expert lectures. Comfort is important, but I'm ready for adventure in this extreme environment.",
        metadata={
            "id": 9,
            "name": "Antarctic Expedition Cruise",
            "weather": "polar",
            "destination": "island",
            "activities": ["adventure", "wildlife"],
            "budget": "ultra_luxury",
            "cost_per_guest": 12999,
            "image": "https://img.freepik.com/free-photo/couple-traveling-together-country-side_23-2149406534.jpg",
            "description": "Explore the icy wilderness of Antarctica 🐧❄️ on an expedition cruise.",
            "sailing dates": ["12/01/2024", "12/15/2024"]
        }
    ),
    Document(
        page_content="We're planning a magical family vacation to Disney World that will create lasting memories for our children. We want a mix of exciting attractions, character meetings, and shows that will keep the kids entertained. Looking for comfortable accommodations with good amenities and meal options suitable for families. We'd like some downtime for the adults while ensuring the children have supervised activities.",
        metadata={
            "id": 10,
            "name": "Family Fun at Disney World",
            "weather": "temperate",
            "destination": "city",
            "activities": ["family", "entertainment"],
            "budget": "standard",
            "cost_per_guest": 2899,
            "image": "https://img.freepik.com/free-photo/full-shot-friends-posing-funfair_23-2148618877.jpg",
            "description": "Enjoy magical moments with the whole family 👨‍👩‍👧‍👦🏰 at Disney World.",
            "sailing dates": ["07/15/2025", "07/22/2025"]
        }
    ),
    Document(
        page_content="I want to explore the stunning Greek islands, hopping from one beautiful location to another. I'm interested in combining beach relaxation with cultural exploration, visiting ancient ruins, trying authentic Greek cuisine, and experiencing traditional village life. Swimming in crystal-clear waters, watching sunsets in Santorini, and learning about Greek mythology would make this trip perfect.",
        metadata={
            "id": 11,
            "name": "Island Hopping in Greece",
            "weather": "temperate",
            "destination": "island",
            "activities": ["relaxation", "cultural"],
            "budget": "standard",
            "cost_per_guest": 2899,
            "image": "https://img.freepik.com/free-photo/curly-short-haired-woman-floral-dress-boater-runs-outside_197531-24118.jpg",
            "description": "Discover the beauty of Greek islands 🏝️ and their rich history 🏛️.",
            "sailing dates": ["06/01/2025", "06/10/2025"]
        }
    ),
    Document(
        page_content="I'm dreaming of an authentic outback adventure in Australia where I can experience the unique landscape and wildlife. I want to explore the red desert, learn about Aboriginal culture, see kangaroos and koalas in their natural habitat, and stay at comfortable but authentic accommodations. Looking for a mix of guided tours and independent exploration.",
        metadata={
            "id": 12,
            "name": "Australian Outback Adventure",
            "weather": "temperate",
            "destination": "countryside",
            "activities": ["adventure", "wildlife"],
            "budget": "standard",
            "cost_per_guest": 3299,
            "image": "https://img.freepik.com/free-photo/sideways-woman-man-waving-each-other-coast_23-2148699842.jpg",
            "description": "Experience the rugged terrain and unique wildlife 🦘🐨 of the Outback.",
            "sailing dates": ["09/15/2025", "09/25/2025"]
        }
    ),
    Document(
        page_content="I want to experience a luxury cruise along the Nile River, exploring ancient Egyptian temples and tombs while enjoying high-end accommodations. I'm fascinated by Egyptian history and archaeology, and would love expert-guided tours of pyramids and ancient sites. Looking for a perfect blend of educational experiences and luxury comfort, with gourmet dining and premium services.",
        metadata={
            "id": 13,
            "name": "Luxury Nile River Cruise",
            "weather": "tropical",
            "destination": "city",
            "activities": ["cultural", "relaxation"],
            "budget": "luxury",
            "cost_per_guest": 6999,
            "image": "https://img.freepik.com/free-photo/dubai-creek_158595-1992.jpg",
            "description": "Sail along the Nile 🚢 and explore ancient Egyptian wonders 🐪🏺.",
            "sailing dates": ["11/01/2025", "11/15/2025"]
        }
    ),
    Document(
        page_content="I'm seeking a transformative wellness experience in the Himalayas, combining traditional healing practices with luxury amenities. I want to practice meditation and yoga with experienced masters, experience traditional therapies, and reconnect with nature in a peaceful mountain setting. Looking for premium accommodation with stunning views and world-class spa facilities.",
        metadata={
            "id": 14,
            "name": "Wellness Spa in the Himalayas",
            "weather": "temperate",
            "destination": "mountain",
            "activities": ["wellness", "relaxation"],
            "budget": "premium",
            "cost_per_guest": 4899,
            "image": "https://img.freepik.com/free-photo/person-practicing-cold-exposure-metabolism_23-2150981869.jpg",
            "description": "Find peace at a spa retreat 🧘‍♂️ nestled in the Himalayas 🏔️.",
            "sailing dates": ["04/05/2025", "04/15/2025"]
        }
    ),
    Document(
        page_content="I dream of experiencing the authentic wine culture and countryside of Tuscany. I want to stay in a beautiful villa, participate in wine tastings at historic vineyards, learn traditional Italian cooking, and explore medieval towns. I'm interested in meeting local winemakers, understanding the wine-making process, and enjoying long, leisurely meals in scenic settings. The combination of culinary experiences and cultural immersion would make this trip perfect.",
        metadata={
            "id": 15,
            "name": "Wine Tasting in Tuscany",
            "weather": "temperate",
            "destination": "countryside",
            "activities": ["cultural", "relaxation"],
            "budget": "premium",
            "cost_per_guest": 4599,
            "image": "https://img.freepik.com/free-photo/low-angle-happy-friends-partying-outdoors_23-2149412443.jpg",
            "description": "Indulge in fine wines 🍷 and picturesque landscapes 🌄 in Tuscany.",
            "sailing dates": ["05/10/2025", "05/20/2025"]
        }
    ),
    Document(
        page_content="I'm passionate about nature and wildlife, and I've always wanted to explore the Amazon rainforest. I want to discover the incredible biodiversity, spot exotic birds and animals, learn about indigenous cultures, and understand rainforest conservation. I'd like to stay in eco-lodges, take guided jungle walks, go on river excursions, and maybe even participate in some conservation activities.",
        metadata={
            "id": 16,
            "name": "Exploring the Amazon Rainforest",
            "weather": "tropical",
            "destination": "countryside",
            "activities": ["adventure", "wildlife"],
            "budget": "standard",
            "cost_per_guest": 3199,
            "image": "https://img.freepik.com/free-photo/young-traveler_1150-5651.jpg",
            "description": "Dive into the heart of the Amazon 🌴 and its diverse ecosystem 🐒🦜.",
            "sailing dates": ["08/05/2025", "08/15/2025"]
        }
    ),
    Document(
        page_content="I want to experience the magical Northern Lights in Iceland with my partner. We're looking for a romantic adventure that combines aurora viewing with exploring Iceland's unique landscape. We'd love to soak in geothermal hot springs, visit ice caves, see volcanic landscapes, and stay in accommodations with glass ceilings for northern lights viewing. A mix of romantic moments and natural wonders would be perfect.",
        metadata={
            "id": 17,
            "name": "Northern Lights in Iceland",
            "weather": "polar",
            "destination": "countryside",
            "activities": ["adventure", "romantic"],
            "budget": "premium",
            "cost_per_guest": 4799,
            "image": "https://img.freepik.com/free-photo/beautiful-aurora-borealis-sky-iceland-spectacular-green-violet-northern-lights-appearing-night-creating-panoramic-landscape-glowing-magical-natural-phenomenon-starry-sky_482257-69775.jpg",
            "description": "Witness the breathtaking Northern Lights 🌠 in Iceland.",
            "sailing dates": ["02/01/2025", "02/10/2025"]
        }
    ),
    Document(
        page_content="I'm looking for a peaceful yoga retreat in Costa Rica where I can deepen my practice and reconnect with nature. I want to practice yoga daily, learn meditation techniques, and enjoy healthy, organic meals. I'd love to combine wellness activities with some adventure like surfing or hiking in the rainforest. Being close to the beach and having access to sustainable, eco-friendly accommodations is important to me.",
        metadata={
            "id": 18,
            "name": "Yoga Retreat in Costa Rica",
            "weather": "tropical",
            "destination": "beach",
            "activities": ["wellness", "relaxation"],
            "budget": "standard",
            "cost_per_guest": 2899,
            "image": "https://img.freepik.com/free-photo/side-view-woman-doing-yoga-nature-with-copy-space_23-2148769597.jpg",
            "description": "Rebalance with yoga sessions 🧘‍♀️ on Costa Rica's serene beaches 🏖️.",
            "sailing dates": ["03/15/2025", "03/25/2025"]
        }
    ),
    Document(
        page_content="I want to immerse myself in the rich history of Rome and explore its ancient wonders. I'm fascinated by Roman history and want to visit all the major archaeological sites, museums, and historical landmarks. I'd love expert-guided tours of the Colosseum, Roman Forum, and Vatican, plus time to explore charming neighborhoods and enjoy authentic Italian cuisine. This would be perfect for our family who loves learning about history together.",
        metadata={
            "id": 19,
            "name": "Historical Tour of Rome",
            "weather": "temperate",
            "destination": "city",
            "activities": ["cultural", "family"],
            "budget": "standard",
            "cost_per_guest": 2999,
            "image": "https://img.freepik.com/free-photo/couple-honeymoon-venice_1303-5723.jpg",
            "description": "Explore ancient ruins 🏛️ and art 🎨 in the heart of Rome.",
            "sailing dates": ["10/05/2025", "10/15/2025"]
        }
    ),
    Document(
        page_content="I'm looking for an exciting party vacation in Ibiza with great beaches and amazing nightlife. I want to experience the best beach clubs, dance at famous nightclubs, enjoy water sports during the day, and meet fellow travelers. I'd love to stay somewhere central with good access to both beaches and nightlife spots. Some cultural activities and island exploration would be great too.",
        metadata={
            "id": 20,
            "name": "Beach Party in Ibiza",
            "weather": "temperate",
            "destination": "beach",
            "activities": ["entertainment", "sports"],
            "budget": "premium",
            "cost_per_guest": 3999,
            "image": "https://img.freepik.com/free-photo/medium-shot-friends-partying-outdoors_23-2149646131.jpg",
            "description": "Enjoy vibrant nightlife 🎉 and water sports 🏄‍♂️ on Ibiza's beaches.",
            "sailing dates": ["07/01/2025", "07/10/2025"]
        }
    ),
    Document(
        page_content="I'd love to explore the Netherlands by bicycle, experiencing the country like a local. I want to cycle through tulip fields, visit historic windmills, explore charming villages, and learn about Dutch culture. I'm interested in sustainable travel and would appreciate staying in eco-friendly accommodations. A mix of urban and rural cycling routes would be perfect, with plenty of cultural stops along the way.",
        metadata={
            "id": 21,
            "name": "Cycling Tour of the Netherlands",
            "weather": "temperate",
            "destination": "countryside",
            "activities": ["sports", "cultural"],
            "budget": "economy",
            "cost_per_guest": 1899,
            "image": "https://img.freepik.com/free-photo/transport-concept-with-people-bicycles_23-2148959676.jpg",
            "description": "Cycle through picturesque landscapes 🚲 and historic towns 🏘️.",
            "sailing dates": ["04/20/2025", "04/30/2025"]
        }
    ),
    Document(
        page_content="I'm eager to discover the unique wildlife and landscapes of Madagascar. I want to see lemurs in their natural habitat, explore unusual geological formations, visit rare baobab trees, and learn about the island's unique ecosystems. I'm interested in both wildlife photography and understanding conservation efforts. Some interaction with local communities and learning about Malagasy culture would make the experience even better.",
        metadata={
            "id": 22,
            "name": "Wildlife Expedition in Madagascar",
            "weather": "tropical",
            "destination": "island",
            "activities": ["wildlife", "adventure"],
            "budget": "standard",
            "cost_per_guest": 3299,
            "image": "https://img.freepik.com/free-photo/beautiful-cheetah-standing-big-branch_181624-18632.jpg",
            "description": "Discover unique species 🦎 on an island like no other 🏝️.",
            "sailing dates": ["09/10/2025", "09/20/2025"]
        }
    ),
    Document(
        page_content="I want to experience the vibrant cultures and traditions of India with my family. We're interested in exploring ancient temples, participating in traditional festivals, learning about Indian spirituality, and trying authentic cuisine from different regions. We'd like to see iconic sites like the Taj Mahal but also experience everyday life in local communities. Looking for a trip that's educational and eye-opening for all ages.",
        metadata={
            "id": 23,
            "name": "Cultural Immersion in India",
            "weather": "tropical",
            "destination": "city",
            "activities": ["cultural", "family"],
            "budget": "economy",
            "cost_per_guest": 1999,
            "image": "https://img.freepik.com/free-photo/man-teaching-children-about-culture-medium-shot_1258-289380.jpg",
            "description": "Experience the diverse cultures and traditions of India 🕌🛕.",
            "sailing dates": ["11/05/2025", "11/15/2025"]
        }
    ),
    Document(
        page_content="I dream of cruising through Norway's magnificent fjords, witnessing some of the world's most dramatic landscapes. I want to sail past towering cliffs, see cascading waterfalls, visit charming coastal villages, and maybe spot some Arctic wildlife. I'd love a mix of on-board relaxation and adventure activities like hiking or kayaking. Learning about Viking history and Norwegian culture would make it even more special.",
        metadata={
            "id": 24,
            "name": "Scandinavian Fjord Cruise",
            "weather": "temperate",
            "destination": "mountain",
            "activities": ["relaxation", "adventure"],
            "budget": "premium",
            "cost_per_guest": 4599,
            "image": "https://img.freepik.com/free-photo/cruise-ship-sea-with-mountains_23-2148153636.jpg",
            "description": "Sail through majestic fjords ⛰️ and enjoy stunning landscapes 🚢.",
            "sailing dates": ["06/20/2025", "06/30/2025"]
        }
    ),
    Document(
        page_content="I'm looking for an extraordinary desert adventure in Dubai that combines luxury with traditional Arabian experiences. I want to go dune bashing in the desert, ride camels at sunset, stay in a luxury desert camp under the stars, and enjoy traditional Bedouin entertainment. I'd also love to experience modern Dubai's attractions, enjoy high-end shopping, and dine at world-class restaurants. The perfect mix of adventure and luxury.",
        metadata={
            "id": 25,
            "name": "Desert Safari in Dubai",
            "weather": "tropical",
            "destination": "countryside",
            "activities": ["adventure", "entertainment"],
            "budget": "luxury",
            "cost_per_guest": 6499,
            "image": "https://img.freepik.com/free-photo/traveling-with-off-road-car_23-2151472970.jpg",
            "description": "Experience dune bashing 🏜️ and cultural shows 🐪 in the desert.",
            "sailing dates": ["10/15/2025", "10/25/2025"]
        }
    )
]
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['f038acaa-71ae-4e4f-918f-9e059a1283e7',
 'fe6d8f64-cb2a-4224-b8d3-aab69bdf5e6c',
 '667bd1ec-b8df-4e62-8858-067533491342',
 '2ba03e93-810c-4401-a2cc-a15f80b3aa11',
 '14315db3-82a2-4e69-a24c-55657ddb06df',
 'ab3992a2-f661-4725-b501-5c882366f3d7',
 '484a3f57-87a7-44be-87ec-2aab8e540231',
 '37c275fb-8481-4d48-b611-bec962633db9',
 'fb49b4e0-1c06-40a9-91fb-df1a2aa9058a',
 '4e686e16-e661-4057-b563-b12a2d14ae46',
 'eec79adc-ab57-4c02-b35e-ca2fe74e75b2',
 'e4378b46-b437-47b3-84ee-eddb9bdef6a9',
 'c56c1989-5b6f-4b40-bf1a-553ecb17ffde',
 'b4fef75d-37b4-47c2-bfa0-60f51fb41504',
 'ec91fe66-0626-485d-b353-31b9207ed5df',
 'e6234d62-e8d8-470c-a78b-9f1508b70bc9',
 '26337f85-9d84-4b96-a250-d9885b35cef0',
 '2b638166-8e30-4586-bc12-54bdf4c387aa',
 '83f7cb3b-5a3f-4eaa-a729-039f9cb3c058',
 '63b79770-4b4f-4b75-b6a0-e805642552fa',
 '105601b0-fefe-466d-bdb3-6bf56b00bc25',
 'e6777da5-1fae-450a-918b-6dc7b4239552',
 '1188f641-7bab-493f-9cc4-fdefc9e91ff0',
 '348c5479-e9aa-48fc-a72b-2789c832a63a',
 '34fcc1b2-5e27-

In [17]:
results = vector_store.similarity_search_with_score(
    "I'm dreaming of a winter wonderland vacation where I can experience the thrill of skiing down snow-covered slopes. I want to enjoy the crisp mountain air and breathtaking views while trying out different ski runs. Cozy accommodations with a warm fireplace and après-ski activities would make this getaway perfect. I'm looking for a destination that offers both challenging slopes for an adrenaline rush and gentle runs for relaxation.", k=25
)
score_threshold = 0.500
for res, score in results:
    if score >= score_threshold:
        print(f"* [SIM={score:.3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.775] I'm looking for a luxurious winter sports vacation in the Swiss Alps. I want to experience world-class skiing during the day and indulge in spa treatments and gourmet dining in the evenings. I'd love to stay in a premium resort with stunning mountain views, access to top-tier skiing facilities, and wellness amenities. The perfect mix of active adventure and refined relaxation. [{'activities': ['sports', 'relaxation'], 'budget': 'luxury', 'cost_per_guest': 6799.0, 'description': 'Hit the slopes ⛷️ and enjoy the scenic beauty of the Swiss Alps 🏔️.', 'destination': 'mountain', 'id': 6.0, 'image': 'https://img.freepik.com/free-photo/beautiful-view-people-cycling-skiing-across-snowy-mountains-south-tyrol-dolomites-italy_181624-29926.jpg', 'name': 'Skiing in the Swiss Alps', 'sailing dates': ['01/15/2025', '01/25/2025'], 'weather': 'temperate'}]
* [SIM=0.687] I'm dreaming of a Caribbean getaway that combines relaxation with adventure. I want to spend my mornings lounging on pri

In [14]:
results

[(Document(id='f038acaa-71ae-4e4f-918f-9e059a1283e7', metadata={'activities': ['relaxation', 'adventure'], 'budget': 'standard', 'cost_per_guest': 2499.0, 'description': 'Enjoy the warm sun ☀️ and beautiful beaches 🏖️ of the Caribbean.', 'destination': 'beach', 'id': 1.0, 'image': 'https://img.freepik.com/free-photo/beautiful-asian-female-woman-relax-casual-leisure-peaceful-moment-cruise-deck-vacation-summer-time_609648-740.jpg', 'name': 'Sunny Caribbean Cruise', 'sailing dates': ['06/15/2025', '06/22/2025'], 'weather': 'tropical'}, page_content="I'm dreaming of a Caribbean getaway that combines relaxation with adventure. I want to spend my mornings lounging on pristine beaches with crystal clear waters, and my afternoons trying exciting water sports or exploring local islands. I'm looking for a balanced vacation that won't break the bank but still offers quality experiences. The warm tropical climate and ocean breezes sound perfect for escaping daily stress."),
  0.526387513),
 (Docum